# Data Cleaning

This notebook performs the cleaning and standardization of the LendingClub dataset.

### Phase 1: Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np

# Load the raw data
raw_data_path = '../data/raw/lending_club_loan_two.csv'
df = pd.read_csv(raw_data_path)

# Integrity Check
df_copy = df.copy()
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (396030, 27)


### Phase 2: Schema Standardization

In [2]:
# 2.1 Column Renaming
df.rename(columns={
    'loan_amnt': 'loan_amount',
    'int_rate': 'interest_rate',
    'installment': 'monthly_installment',
    'annual_inc': 'annual_income',
    'dti': 'debt_to_income_ratio',
    'open_acc': 'number_of_open_accounts',
    'pub_rec': 'public_records_derogatory',
    'revol_bal': 'revolving_balance',
    'revol_util': 'revolving_line_utilization',
    'total_acc': 'total_credit_lines',
    'pub_rec_bankruptcies': 'public_record_bankruptcies',
    'issue_d': 'issue_date',
    'earliest_cr_line': 'earliest_credit_line_opened',
    'term': 'loan_term',
    'grade': 'loan_grade',
    'sub_grade': 'loan_sub_grade',
    'home_ownership': 'home_ownership_status',
    'verification_status': 'income_verification_status',
    'loan_status': 'loan_repayment_status',
    'purpose': 'loan_purpose',
    'title': 'loan_title_description',
    'zip_code': 'zip_code_prefix',
    'addr_state': 'borrower_state_address',
    'initial_list_status': 'initial_listing_status',
    'application_type': 'loan_application_type',
    'address': 'borrower_full_address',
    'log_income': 'log_annual_income',
    'mort_acc':'number_of_mortgage_accounts'
}, inplace=True)

# 2.2 Verification
print("Columns after renaming:")
print(df.columns)

Columns after renaming:
Index(['loan_amount', 'loan_term', 'interest_rate', 'monthly_installment',
       'loan_grade', 'loan_sub_grade', 'emp_title', 'emp_length',
       'home_ownership_status', 'annual_income', 'income_verification_status',
       'issue_date', 'loan_repayment_status', 'loan_purpose',
       'loan_title_description', 'debt_to_income_ratio',
       'earliest_credit_line_opened', 'number_of_open_accounts',
       'public_records_derogatory', 'revolving_balance',
       'revolving_line_utilization', 'total_credit_lines',
       'initial_listing_status', 'loan_application_type',
       'number_of_mortgage_accounts', 'public_record_bankruptcies',
       'borrower_full_address'],
      dtype='object')


### Phase 3: Structural Cleaning

In [3]:
# 3.1 Drop Redundant Columns
df.drop(['emp_title', 'emp_length', 'number_of_mortgage_accounts'], axis=1, inplace=True)

# 3.2 Handle Duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicates found: {duplicates_count}")
df.drop_duplicates(inplace=True)

Number of duplicates found: 0


### Phase 4: Missing Value Treatment

In [4]:
# 4.1 Numeric Imputation (Median)
df['revolving_line_utilization'].fillna(df['revolving_line_utilization'].median(), inplace=True)
df['public_record_bankruptcies'].fillna(df['public_record_bankruptcies'].median(), inplace=True)

# 4.2 Categorical Imputation
df['loan_title_description'].fillna("Unknown", inplace=True)

# 4.3 Final Removal of remaining nulls
df.dropna(inplace=True)

print("Null values after treatment:")
print(df.isnull().sum())

/var/folders/sl/wct_4c450z95hfy7x_xgm8vr0000gn/T/ipykernel_94577/1864455378.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['revolving_line_utilization'].fillna(df['revolving_line_utilization'].median(), inplace=True)
/var/folders/sl/wct_4c450z95hfy7x_xgm8vr0000gn/T/ipykernel_94577/1864455378.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interme

Null values after treatment:
loan_amount                    0
loan_term                      0
interest_rate                  0
monthly_installment            0
loan_grade                     0
loan_sub_grade                 0
home_ownership_status          0
annual_income                  0
income_verification_status     0
issue_date                     0
loan_repayment_status          0
loan_purpose                   0
loan_title_description         0
debt_to_income_ratio           0
earliest_credit_line_opened    0
number_of_open_accounts        0
public_records_derogatory      0
revolving_balance              0
revolving_line_utilization     0
total_credit_lines             0
initial_listing_status         0
loan_application_type          0
public_record_bankruptcies     0
borrower_full_address          0
dtype: int64


### Phase 5: Data Type & Outlier Cleaning

In [5]:
# 5.1 Type Casting
df['loan_amount'] = df['loan_amount'].astype(float)
df['annual_income'] = df['annual_income'].astype(float)

# 5.2 DateTime Parsing
df['issue_date'] = pd.to_datetime(df['issue_date'], format='%b-%Y')

# 5.3 String Trimming
df['loan_term'] = df['loan_term'].str.strip()

# 5.4 Outlier Filtering (Annual Income < 99th percentile)
df = df[df['annual_income'] < df['annual_income'].quantile(0.99)]

print("Final Data Types:")
print(df.dtypes)

Final Data Types:
loan_amount                           float64
loan_term                              object
interest_rate                         float64
monthly_installment                   float64
loan_grade                             object
loan_sub_grade                         object
home_ownership_status                  object
annual_income                         float64
income_verification_status             object
issue_date                     datetime64[ns]
loan_repayment_status                  object
loan_purpose                           object
loan_title_description                 object
debt_to_income_ratio                  float64
earliest_credit_line_opened            object
number_of_open_accounts               float64
public_records_derogatory             float64
revolving_balance                     float64
revolving_line_utilization            float64
total_credit_lines                    float64
initial_listing_status                 object
loan_application

### Phase 6: Quality Audit & Export

In [6]:
# Final Statistics and Validation
print("--- Final Dataset Info ---")
df.info()

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics ---")
display(df.describe())

print(f"\nFinal dataset shape: {df.shape}")

--- Final Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 391953 entries, 0 to 396029
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   loan_amount                  391953 non-null  float64       
 1   loan_term                    391953 non-null  object        
 2   interest_rate                391953 non-null  float64       
 3   monthly_installment          391953 non-null  float64       
 4   loan_grade                   391953 non-null  object        
 5   loan_sub_grade               391953 non-null  object        
 6   home_ownership_status        391953 non-null  object        
 7   annual_income                391953 non-null  float64       
 8   income_verification_status   391953 non-null  object        
 9   issue_date                   391953 non-null  datetime64[ns]
 10  loan_repayment_status        391953 non-null  object        
 11  loan

,loan_amount,interest_rate,monthly_installment,annual_income,issue_date,debt_to_income_ratio,number_of_open_accounts,public_records_derogatory,revolving_balance,revolving_line_utilization,total_credit_lines,public_record_bankruptcies
count,391953.000000,391953.000000,391953.000000,391953.000000,391953,391953.000000,391953.000000,391953.000000,391953.000000,391953.000000,391953.000000,391953.000000
mean,14002.558598,13.645649,428.356682,71062.355368,2014-02-01 17:34:56.470750464,17.461062,11.292249,0.178180,15421.266134,53.781989,25.349118,0.122316
min,500.000000,5.320000,16.080000,0.000000,2007-06-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000
25%,7875.000000,10.490000,249.540000,45000.000000,2013-05-01 00:00:00,11.390000,8.000000,0.000000,6001.000000,35.900000,17.000000,0.000000
50%,12000.000000,13.330000,373.600000,63000.000000,2014-04-01 00:00:00,16.990000,10.000000,0.000000,11115.000000,54.800000,24.000000,0.000000
75%,19675.000000,16.550000,562.700000,88300.000000,2015-03-01 00:00:00,23.040000,14.000000,0.000000,19425.000000,72.800000,32.000000,0.000000
max,40000.000000,30.990000,1527.000000,249999.960000,2016-12-01 00:00:00,9999.000000,90.000000,86.000000,814300.000000,892.300000,151.000000,8.000000
std,8273.385068,4.468015,247.610255,36426.359312,NaN,18.084559,5.125592,0.528923,17788.221336,24.411770,11.855683,0.357064



Final dataset shape: (391953, 24)


### Phase 7: Feature Engineering

In [7]:
# ------------------------------------------------
# 1. loan_income_ratio
# Meaning: Loan amount compared to the borrower's yearly income.
# Purpose: Shows how large the loan is relative to income.
# Higher value means the borrower is taking a bigger loan burden.
# ------------------------------------------------
df["loan_income_ratio"] = df["loan_amount"] / df["annual_income"]


# ------------------------------------------------
# 2. installment_income_ratio
# Meaning: Monthly installment compared to monthly income.
# Purpose: Measures how much of the borrower's monthly income
# goes toward loan repayment.
# Higher value means higher monthly repayment pressure.
# ------------------------------------------------
df["installment_income_ratio"] = (
    df["monthly_installment"] / (df["annual_income"] / 12)
)


# ------------------------------------------------
# 3. risk_score
# Meaning: Combined risk indicator using DTI, utilization,
# loan-income ratio, and bankruptcy records.
# Purpose: Gives a single numeric score to compare borrower risk.
# Higher score means the borrower may be financially riskier.
# ------------------------------------------------
df["risk_score"] = (
    0.4 * df["debt_to_income_ratio"] +
    0.3 * df["revolving_line_utilization"] +
    0.2 * df["loan_income_ratio"] * 100 +
    0.1 * df["public_record_bankruptcies"].fillna(0) * 10
)


# ------------------------------------------------
# 4. credit_stress_score
# Meaning: Measures credit pressure using DTI, credit utilization,
# and derogatory public records.
# Purpose: Helps identify borrowers under financial stress.
# Higher score means higher credit stress.
# ------------------------------------------------
df["credit_stress_score"] = (
    df["debt_to_income_ratio"] +
    df["revolving_line_utilization"] +
    (df["public_records_derogatory"].fillna(0) * 10)
)


# ------------------------------------------------
# 5. issue_year
# Meaning: Year in which the loan was issued.
# Purpose: Useful for year-wise trend analysis such as
# loan volume, average loan amount, interest rate, or risk over time.
# ------------------------------------------------
df["issue_date"] = pd.to_datetime(
    df["issue_date"].astype(str),
    format="%b-%Y",
    errors="coerce"
)

df["issue_year"] = df["issue_date"].dt.year


# ------------------------------------------------
# 6. income_band
# Meaning: Groups borrowers into income categories.
# Purpose: Makes income-based EDA easier by comparing
# low, middle, and high income borrowers.
# ------------------------------------------------
df["income_band"] = pd.cut(
    df["annual_income"],
    bins=[0, 40000, 100000, np.inf],
    labels=["Low", "Middle", "High"]
)


# ------------------------------------------------
# 7. dti_band
# Meaning: Groups borrowers based on debt-to-income ratio.
# Purpose: Helps compare borrowers with low, medium,
# and high debt burden.
# ------------------------------------------------
df["dti_band"] = pd.cut(
    df["debt_to_income_ratio"],
    bins=[0, 10, 20, np.inf],
    labels=["Low", "Medium", "High"]
)


# ------------------------------------------------
# 8. utilization_band
# Meaning: Groups borrowers based on revolving credit utilization.
# Purpose: Helps compare borrowers with low, medium,
# and high credit usage.
# Higher utilization usually means higher financial pressure.
# ------------------------------------------------
df["utilization_band"] = pd.cut(
    df["revolving_line_utilization"],
    bins=[0, 30, 70, 100],
    labels=["Low", "Medium", "High"]
)


df.head()

# Export Cleaned Data
output_path = '../data/processed/cleaned_club_loan_two.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved successfully to: {output_path}")


Cleaned dataset saved successfully to: ../data/processed/cleaned_club_loan_two.csv
